In [11]:
!pip install -q pygam

In [2]:
import time
import warnings
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from pygam import LinearGAM, s, te
from pygam.terms import TermList
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/shauryapandey19/pm25-dataset/delhi_33station_pm25_spatiotemporal_dataset_2018_2026.csv"

use_columns = [
    "timestamp_ist", "station_name", "latitude", "longitude", "pm25", "pm25_original",
    "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos", "boundary_layer_height_m",
    "fire_frp_sum", "fire_count",
]

df = pd.read_csv(DATA_PATH, usecols=use_columns, parse_dates=["timestamp_ist"])
df = df.rename(columns={"timestamp_ist": "timestamp", "station_name": "station_id"})
df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

print(df.shape)
print(df["station_id"].nunique())

# the station-by-hour grids need unique (station, hour) pairs
assert not df.duplicated(subset=["station_id", "timestamp"]).any()

(2392598, 12)
33


In [3]:
# prep already set PM2.5 <= 0 to NaN, this only verifies it (NaN compares False, so nulls pass)
assert df["pm25"].notna().all() and (df["pm25"] > 0).all()
assert not (df["pm25_original"] <= 0).any()

print("pm25_original nulls (interpolated hours):", int(df["pm25_original"].isna().sum()))

pm25_original nulls (interpolated hours): 25674


In [4]:
df["ist_date"] = df["timestamp"].dt.normalize()

fire_daily_columns = ["fire_frp_sum", "fire_count"]

# fire values are one per (station, IST day), repeated on every hour of that day
assert df.groupby(["station_id", "ist_date"])[fire_daily_columns].nunique().to_numpy().max() == 1

daily_fire = df.drop_duplicates(["station_id", "ist_date"])[["station_id", "ist_date"] + fire_daily_columns]

for lag_days in [1, 2, 3]:
    lag_table = daily_fire.copy()
    # moving the date forward puts day D's value on day D + lag_days
    lag_table["ist_date"] = lag_table["ist_date"] + pd.Timedelta(days=lag_days)
    lag_table = lag_table.rename(columns={
        "fire_frp_sum": "fire_frp_lag" + str(lag_days),
        "fire_count": "fire_count_lag" + str(lag_days),
    })
    df = df.merge(lag_table, on=["station_id", "ist_date"], how="left")

df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

# log1p because FRP and counts are very skewed, a lag is NaN when its day is missing
df["fire_frp_today_log"] = np.log1p(df["fire_frp_sum"])
df["fire_frp_past3_log"] = np.log1p((df["fire_frp_lag1"] + df["fire_frp_lag2"] + df["fire_frp_lag3"]) / 3)
df["fire_count_today_log"] = np.log1p(df["fire_count"])
df["fire_count_past3_log"] = np.log1p((df["fire_count_lag1"] + df["fire_count_lag2"] + df["fire_count_lag3"]) / 3)

fire_feature_columns = ["fire_frp_today_log", "fire_frp_past3_log", "fire_count_today_log", "fire_count_past3_log"]

In [5]:
station_list = sorted(df["station_id"].unique())
n_stations = len(station_list)
station_to_idx = {name: i for i, name in enumerate(station_list)}
s_idx = df["station_id"].map(station_to_idx).to_numpy()

# complete hourly index, so shifting the grid by n rows is exactly n hours
full_index = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
n_hours = len(full_index)
t_idx = full_index.get_indexer(df["timestamp"])
assert (t_idx >= 0).all()

# pm25_grid[t, i] = pm25 of station i at hour t, NaN where the station has no row
pm25_grid = np.full((n_hours, n_stations), np.nan)
pm25_grid[t_idx, s_idx] = df["pm25"].to_numpy()

LAG_HOURS = [0, 1, 3, 6]
value_grids = []
avail_grids = []
for lag in LAG_HOURS:
    shifted = np.full_like(pm25_grid, np.nan)
    shifted[lag:] = pm25_grid[:n_hours - lag]
    available = ~np.isnan(shifted)
    # zero-filled values and 0/1 availability turn every IDW sum into a matrix product
    value_grids.append(np.where(available, shifted, 0.0))
    avail_grids.append(available.astype(np.float64))

print(n_stations, n_hours, pm25_grid.shape)

33 76680 (76680, 33)


In [6]:
station_coords = df.drop_duplicates("station_id").set_index("station_id").loc[station_list, ["latitude", "longitude"]]
lat_r = np.radians(station_coords["latitude"].to_numpy())
lon_r = np.radians(station_coords["longitude"].to_numpy())

# haversine km, dist_matrix[j, i] = target j to source i
a_term = (np.sin((lat_r[None, :] - lat_r[:, None]) / 2) ** 2
          + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.sin((lon_r[None, :] - lon_r[:, None]) / 2) ** 2)
dist_matrix = 2 * 6371.0 * np.arcsin(np.sqrt(a_term))

IDW_POWER = 1.5
MIN_DIST_KM = 0.5

# W[j, i] = weight of source i for target j, floor stops near-identical coordinates getting huge weight
W = 1.0 / np.maximum(dist_matrix, MIN_DIST_KM) ** IDW_POWER
np.fill_diagonal(W, 0.0)

In [7]:
hour = df["timestamp"].dt.hour.to_numpy()
day_of_year = df["timestamp"].dt.dayofyear.to_numpy()

# columns 0-6 of the GAM input: trend (years), hour, season position (0 to 1), BLH, wind speed, wind sin, wind cos
X_static = np.column_stack([
    t_idx / 8760.0,
    hour,
    (day_of_year - 1 + hour / 24.0) / 366.0,
    df["boundary_layer_height_m"].to_numpy(),
    df["wind_speed_m_s"].to_numpy(),
    df["wind_dir_sin"].to_numpy(),
    df["wind_dir_cos"].to_numpy(),
])

fire_X = df[fire_feature_columns].to_numpy(dtype=np.float64)
y_all = df["pm25_original"].to_numpy(dtype=np.float64)

# boundary_layer_height_m is null for 2024-01-01 to 2024-06-30, those rows drop out here
required_columns = ["pm25_original", "boundary_layer_height_m", "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos"]
row_ok = df[required_columns].notna().all(axis=1).to_numpy()

shared = {
    "V": value_grids, "A": avail_grids, "W": W,
    "t_idx": t_idx, "s_idx": s_idx,
    "X_static": X_static, "fire_X": fire_X, "y": y_all, "row_ok": row_ok,
}

print(int(row_ok.sum()), len(df))

2230464 2392598


In [9]:
LAM_GRID = [1, 10, 100, 1000]

# whole stations go to validation, and they are removed from the neighbour network inside run_split
cv_splits = list(GroupKFold(n_splits=3).split(np.arange(n_stations), groups=np.arange(n_stations)))

tune_jobs = []
for lam in LAM_GRID:
    for train_stations, val_stations in cv_splits:
        tune_jobs.append((lam, val_stations))

tune_out = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(run_split)(shared, job[1], job[0], TRAIN_STRIDE, TRAIN_STRIDE, False) for job in tune_jobs
)

mae_by_lam = {}
for job_pos in range(len(tune_jobs)):
    lam = tune_jobs[job_pos][0]
    rows, pred, _ = tune_out[job_pos]
    mae_by_lam.setdefault(lam, []).append(np.mean(np.abs(y_all[rows] - pred)))

for lam in LAM_GRID:
    print("lam", lam, "grouped-CV MAE", round(float(np.mean(mae_by_lam[lam])), 3))

best_lam = min(LAM_GRID, key=lambda lam: np.mean(mae_by_lam[lam]))
print("best lam:", best_lam)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   8 out of  12 | elapsed:  2.9min remaining:  1.4min


lam 1 grouped-CV MAE 23.497
lam 10 grouped-CV MAE 23.49
lam 100 grouped-CV MAE 23.481
lam 1000 grouped-CV MAE 23.471
best lam: 1000


[Parallel(n_jobs=4)]: Done  12 out of  12 | elapsed:  4.3min finished


In [13]:
import gc

# hourly rows are strongly autocorrelated, so training on every 6th hour keeps fits fast with little loss
# every 12th hour halves the training rows and the dense matrices pyGAM builds
TRAIN_STRIDE = 12
N_JOBS = 3
BASE_COLS = 11

# feature index: 0 trend, 1 hour, 2 season, 3 BLH, 4 wind speed, 5-6 wind sin/cos, 7-10 IDW at t, t-1h, t-3h, t-6h, 11-14 fire
def build_terms(lam, with_fire):
    terms = [
        s(0, lam=lam),
        # cyclic so hour 24 joins hour 0
        s(1, basis="cp", edge_knots=[0, 24], lam=lam),
        # cyclic so 31 Dec joins 1 Jan
        s(2, basis="cp", edge_knots=[0, 1], lam=lam),
        s(3, lam=lam),
        s(4, lam=lam),
        te(5, 6, lam=lam),
    ]
    for col in [7, 8, 9, 10]:
        terms.append(s(col, lam=lam))
    if with_fire:
        for col in [11, 12, 13, 14]:
            terms.append(s(col, lam=lam))
    return TermList(*terms)


def run_split(shared, removed, lam, train_stride, test_stride, with_fire):
    warnings.filterwarnings("ignore")
    t_idx = shared["t_idx"]
    s_idx = shared["s_idx"]
    n_rows = len(t_idx)

    # removed stations stop being sources for every row, train and test
    W = shared["W"].copy()
    W[:, removed] = 0.0

    nb = np.empty((n_rows, 4))
    usable = shared["row_ok"].copy()
    for k in range(4):
        # (V @ W.T)[t, j] = sum over sources i of V[t, i] * W[j, i]
        num = shared["V"][k] @ W.T
        den = shared["A"][k] @ W.T
        den_rows = den[t_idx, s_idx]
        # a row needs at least one other source at t, t-1h, t-3h and t-6h, same rule as the other models
        usable &= den_rows > 0
        nb[:, k] = num[t_idx, s_idx] / np.where(den_rows > 0, den_rows, 1.0)

    removed_rows = np.isin(s_idx, removed)
    train_rows = usable & ~removed_rows & (t_idx % train_stride == 0)
    test_rows = usable & removed_rows & (t_idx % test_stride == 0)

    X_train = np.column_stack([shared["X_static"][train_rows], nb[train_rows], shared["fire_X"][train_rows]])
    X_test = np.column_stack([shared["X_static"][test_rows], nb[test_rows], shared["fire_X"][test_rows]])
    y_train = shared["y"][train_rows]

    gam = LinearGAM(build_terms(lam, False), max_iter=100)
    gam.fit(np.ascontiguousarray(X_train[:, :BASE_COLS]), y_train)
    # true PM2.5 is positive, so negative predictions are clipped to 0
    pred = np.clip(gam.predict(np.ascontiguousarray(X_test[:, :BASE_COLS])), 0.0, None)

    pred_fire = np.full(len(pred), np.nan)
    if with_fire:
        # fire lags are NaN on a few early or gap days, the fire model only uses rows with all four finite
        fire_train = np.isfinite(X_train[:, BASE_COLS:]).all(axis=1)
        fire_test = np.isfinite(X_test[:, BASE_COLS:]).all(axis=1)
        gam_fire = LinearGAM(build_terms(lam, True), max_iter=100)
        gam_fire.fit(X_train[fire_train], y_train[fire_train])
        pred_fire[fire_test] = np.clip(gam_fire.predict(X_test[fire_test]), 0.0, None)

    del X_train, X_test, gam
    if with_fire:
        del gam_fire
    gc.collect()

    return np.flatnonzero(test_rows), pred, pred_fire

In [14]:
from joblib.externals.loky import get_reusable_executor

BATCH_SIZE = 4
t_start = time.perf_counter()

loso_out = []
for start in range(0, n_stations, BATCH_SIZE):
    batch = range(start, min(start + BATCH_SIZE, n_stations))
    batch_out = Parallel(n_jobs=N_JOBS)(
        delayed(run_split)(shared, [held_idx], best_lam, TRAIN_STRIDE, 1, True) for held_idx in batch
    )
    loso_out += batch_out

    # shutting the workers down returns their memory to the OS, loky otherwise keeps it
    get_reusable_executor().shutdown(wait=True)
    print("folds done:", len(loso_out), f"| {(time.perf_counter() - t_start) / 60:.1f} min")

folds done: 4 | 2.9 min
folds done: 8 | 5.9 min
folds done: 12 | 8.9 min
folds done: 16 | 11.9 min
folds done: 20 | 14.9 min
folds done: 28 | 20.8 min
folds done: 32 | 23.8 min
folds done: 33 | 25.0 min


In [15]:
timestamps = df["timestamp"].to_numpy()

frames = []
for held_idx in range(n_stations):
    rows, pred, pred_fire = loso_out[held_idx]
    frames.append(pd.DataFrame({
        "station_id": station_list[held_idx],
        "timestamp": timestamps[rows],
        "actual": y_all[rows],
        "gam_predicted": pred,
        "gam_fire_predicted": pred_fire,
    }))

loso_df = pd.concat(frames, ignore_index=True)
loso_df.to_csv("/kaggle/working/loso_df_gam_33station.csv", index=False)

print(loso_df.shape)
print("rows with fire predictions:", int(loso_df["gam_fire_predicted"].notna().sum()))

(2229994, 5)
rows with fire predictions: 2205877


In [16]:
def calculate_metrics(actual_values, predicted_values):
    actual_values = np.asarray(actual_values, dtype=np.float64)
    predicted_values = np.asarray(predicted_values, dtype=np.float64)
    errors = actual_values - predicted_values

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    r2 = 1 - np.sum(errors ** 2) / np.sum((actual_values - np.mean(actual_values)) ** 2)

    # positive means over-prediction
    bias = np.mean(predicted_values - actual_values)
    return mae, rmse, r2, bias


def report(label, data):
    common = data.dropna(subset=["gam_fire_predicted"])
    print("-----", label, "| n =", len(data), "| n with fire features =", len(common))
    mae, rmse, r2, bias = calculate_metrics(data["actual"], data["gam_predicted"])
    print(f"gam, all rows          MAE {mae:.3f}  RMSE {rmse:.2f}  R2 {r2:.3f}  bias {bias:.2f}")
    for col in ["gam_predicted", "gam_fire_predicted"]:
        mae, rmse, r2, bias = calculate_metrics(common["actual"], common[col])
        print(f"{col:22s} MAE {mae:.3f}  RMSE {rmse:.2f}  R2 {r2:.3f}  bias {bias:.2f}  (fire rows)")


report("all rows", loso_df)

----- all rows | n = 2229994 | n with fire features = 2205877
gam, all rows          MAE 24.283  RMSE 42.62  R2 0.811  bias -2.44
gam_predicted          MAE 24.289  RMSE 42.54  R2 0.812  bias -2.44  (fire rows)
gam_fire_predicted     MAE 24.223  RMSE 42.48  R2 0.812  bias -2.24  (fire rows)


In [17]:
per_station_rows = []
for station, station_rows in loso_df.groupby("station_id"):
    row = {"station_id": station, "n_rows": len(station_rows)}

    mae, rmse, r2, bias = calculate_metrics(station_rows["actual"], station_rows["gam_predicted"])
    row["gam_mae"] = mae
    row["gam_rmse"] = rmse
    row["gam_r2"] = r2
    row["gam_bias"] = bias

    # fire vs no-fire compared on the same rows
    common = station_rows.dropna(subset=["gam_fire_predicted"])
    row["gam_mae_common"] = calculate_metrics(common["actual"], common["gam_predicted"])[0]
    row["fire_mae_common"] = calculate_metrics(common["actual"], common["gam_fire_predicted"])[0]
    per_station_rows.append(row)

per_station_df = pd.DataFrame(per_station_rows)
print(per_station_df.round(3).to_string())

print("fire lowers MAE at", int((per_station_df["fire_mae_common"] < per_station_df["gam_mae_common"]).sum()),
      "of", len(per_station_df), "stations")

                            station_id  n_rows  gam_mae  gam_rmse  gam_r2  gam_bias  gam_mae_common  fire_mae_common
0                               alipur   61510   19.888    32.887   0.883     1.835          19.873           19.812
1                          anand_vihar   65681   31.763    54.415   0.786   -27.061          31.713           31.555
2                          ashok_vihar   68243   16.991    30.770   0.919    -2.796          16.985           16.920
3                       aurobindo_marg   66207   20.038    33.299   0.821    14.355          20.101           20.168
4                            aya_nagar   66874   31.680    48.651   0.566    19.301          31.525           31.595
5                               bawana   65523   28.309    48.575   0.805   -19.123          28.366           28.195
6                    crri_mathura_road   68115   22.589    37.753   0.824     3.754          22.606           22.513
7        dr_karni_singh_shooting_range   66524   15.167    25.97

In [18]:
block_start = pd.Timestamp("2025-10-15")
block_end = pd.Timestamp("2025-12-01")
window_mask = (loso_df["timestamp"] >= block_start) & (loso_df["timestamp"] < block_end)

report("actual < 100", loso_df[loso_df["actual"] < 100])
report("actual >= 100", loso_df[loso_df["actual"] >= 100])
report("Oct 15-Nov 30 2025", loso_df[window_mask])

----- actual < 100 | n = 1441564 | n with fire features = 1423432
gam, all rows          MAE 14.106  RMSE 22.55  R2 0.133  bias 4.42
gam_predicted          MAE 14.106  RMSE 22.56  R2 0.133  bias 4.43  (fire rows)
gam_fire_predicted     MAE 14.049  RMSE 22.52  R2 0.136  bias 4.60  (fire rows)
----- actual >= 100 | n = 788430 | n with fire features = 782445
gam, all rows          MAE 42.890  RMSE 64.86  R2 0.603  bias -14.99
gam_predicted          MAE 42.814  RMSE 64.63  R2 0.606  bias -14.93  (fire rows)
gam_fire_predicted     MAE 42.732  RMSE 64.53  R2 0.607  bias -14.69  (fire rows)
----- Oct 15-Nov 30 2025 | n = 36426 | n with fire features = 36370
gam, all rows          MAE 44.895  RMSE 69.73  R2 0.601  bias -2.77
gam_predicted          MAE 44.888  RMSE 69.71  R2 0.601  bias -2.73  (fire rows)
gam_fire_predicted     MAE 44.925  RMSE 69.72  R2 0.601  bias -2.98  (fire rows)


In [19]:
loso_df["gam_sq_error"] = (loso_df["actual"] - loso_df["gam_predicted"]) ** 2

share_df = loso_df.groupby("station_id")["gam_sq_error"].agg(["size", "sum"]).reset_index()
share_df["pct_of_rows"] = 100 * share_df["size"] / len(loso_df)
share_df["pct_of_sq_error"] = 100 * share_df["sum"] / loso_df["gam_sq_error"].sum()
share_df = share_df.drop(columns="sum").rename(columns={"size": "n_rows"})

print(share_df.sort_values("pct_of_sq_error", ascending=False).round(2).to_string())

                            station_id  n_rows  pct_of_rows  pct_of_sq_error
28                            shadipur   68763         3.08             5.97
10                ihbas_dilshad_garden   67578         3.03             5.89
21                                nsut   69256         3.11             5.78
19                         nehru_nagar   69743         3.13             5.68
12                        jahangirpuri   68557         3.07             4.97
8                                  dtu   66459         2.98             4.84
1                          anand_vihar   65681         2.95             4.80
16                              mundka   65579         2.94             4.20
31                         vivek_vihar   68371         3.07             4.12
24                        punjabi_bagh   68883         3.09             3.94
4                            aya_nagar   66874         3.00             3.91
5                               bawana   65523         2.94             3.82